# **Note for Colab Users**

# **Do not write directly in this file. Your work may disappear!**

# **Always make a copy before you start.**

How to make a copy

1. Click **File** in the top left  
> *If you cannot see menus like **File** or **Runtime**, click the **v** mark in the top right to show them.*

2. Choose **Save a copy in Drive**  

3. Rename the copied file to `YOURNAMEs_FileName.ipynb`  
> Example: if your name is Olivia → Olivias_FileName.ipynb  


---

* Checkmarks (✅) are not saved. They disappear when you refresh the page with Chrome's reload button.<br>  
If you stop halfway, add a text cell and write something like `SO FAR DONE`.

---

* In Colab, **previous outputs are reset every 30 to 90 minutes**.<br>  
Because of that, errors like `~~ is not defined` happen **all the time**.

🔁 What should you do if you get a `~~ is not defined` error?

1. First, check the spelling of the variable name.<br>  
2. If the spelling is correct but the error still appears, **click that cell to select it**.<br>  
3. Click **Runtime** → **Run before** in the top left.<br>  
→ This reruns **all cells before that cell**.  
4. Run that cell again.

If the error still does not go away,<br>  
there may be a basic mistake in an earlier TODO answer.<br>  
Check whether it is correct.<br>  
Or ask ChatGPT or another coding assistant for help.


In [ ]:
# A Function to Display Tensors Nicely (Feel Free to Skip This)
import torch
import torch.nn as nn
import torch.nn.functional as F

# Install the library that displays tensors nicely
!pip install git+https://github.com/HayatoHongo/print_formatted_tensor.git
# Import the function that displays PyTorch tensors nicely
from torch_print_tensor import print_formatted_tensor


# **Chapter 19: Long Train**

### **Section 1: Converting to an npy File**


Last time, we could train for only 1 hour, so training stopped before the validation loss and training loss had fully flattened out.

This time, we will train the model properly for about 6 hours.

At the same time, to avoid putting pressure on system RAM, we will change the data sampling method to index access.

With longer training, the model will move a little closer to ChatGPT.

Let's get started.


In [ ]:
data_file = "https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu/resolve/main/sample/10BT/000_00000.parquet"

```python
function: load_dataset
arguments:
path: "parquet",
data_files: THINK_BY_YOURSELF
split : "train"
cache_dir : "/content/fineweb"
```

* `load_dataset` is a function from the 🤗 Hugging Face Datasets library that loads a dataset in the specified format.
* `path="parquet"` means the file to load is in **Parquet format**.
* `data_files=data_file` specifies the target file to load, either a file path or a list.
* `split="train"` specifies the **split**, such as train or test.
* `cache_dir="/content/fineweb"` specifies the directory where the downloaded dataset is cached.


In [ ]:
from datasets import load_dataset
ds = # TODO: function(arguments)

`ds` is now an instance of HuggingFace Datasets' special `Dataset` class.


In [ ]:
ds

In [ ]:
# Document boundary
boundary = "<|endoftext|>"

with open("fineweb.txt", "w", encoding="utf-8") as f:
    for sample in ds:
        f.write(boundary + sample["text"])

#### ⚡ Efficient Training Data Preparation on a T4 GPU

This time, we use a T4 GPU and train for about 6 hours.  
To secure enough data, we will use a text file of about 3.3GB, `fineweb.txt`.

---

🚨 Problem: RAM pressure

Until now, we loaded the entire text into memory at once,  
and then sampled from it randomly.  

But when the file size exceeds 3GB, this method can use up the system RAM, about 12 to 16GB, and crash the session.

---

💡 Solution: NumPy memory mapping plus index access

To avoid stressing RAM, we use NumPy's memory-map feature, `mmap_mode="r"`.  
This lets us read only the necessary parts by index without expanding the whole file in memory.
  
“Index access” means reading only specific positions, or indices, from a huge file.   

This keeps RAM usage low while still allowing fast reads.

---

⚙️ Processing Flow

1. Convert text ahead of time with `tiktoken` into a sequence of token IDs, which are integers  
2. Save the token sequence in `int32` format as a binary file, `.bin`  
3. Convert `.bin` to NumPy format, `.npy`  
4. Load it with memory mapping using `np.load(..., mmap_mode="r")`

With this setup, we can read only the needed parts without loading the entire file into RAM.


1. Convert text ahead of time with `tiktoken` into a sequence of token IDs, which are integers  


In [ ]:
import numpy as np
import tiktoken
import os

In [ ]:
# Specify the file path
input_text_path = "fineweb.txt"

```python
function: tiktoken.get_encoding
arguments: encoding name, such as "gpt2"
```

* `tiktoken.get_encoding("gpt2")` gets the **GPT-2 tokenizer, or encoder**, from OpenAI's `tiktoken` library.
* It is used to perform token conversion for GPT-style models.
* The encoder you get can convert text into token ID sequences and convert them back.


In [ ]:
# Prepare the tokenizer
encoder = # TODO: function(arguments)

The file contains hundreds of millions of characters. We encode these characters into a token sequence.


```python
function: open
arguments: file = path to the text file, THINK_BY_YOURSELF, mode = "r", encoding="utf-8"
```

* Opens the text file at the specified path in **read mode ("r")**.
* `encoding="utf-8"` sets the character encoding to UTF-8.
* The return value is a file object, which can be used for reading and writing.


In [ ]:
text_file = # TODO: function(arguments)

`fineweb.txt` contains hundreds of millions of characters. If we try to encode all of it at once, system RAM will crash.

So we cut out fixed-length strings little by little. Each block is called a chunk.

We encode each chunk. This lets us process a huge file gradually.




```python
instance: text_file
method: THINK_BY_YOURSELF
arguments: 100, the number of characters to read
```

* Reads only 100 characters from the file object `text_file`.
* The loaded string is stored in the variable `chunk`.


In [ ]:
chunk = # TODO: instance.method(arguments)
print(chunk)

```python
instance: text_file
method: close
arguments: none
```

* `text_file.close()` is a method for closing an open file.
* It is important to call `close()` after file operations to release resources.
* Especially when you open a file with `open()`, you need to explicitly close it.


In [ ]:
# TODO: instance.method(arguments)

```python
instance: encoder
method: THINK_BY_YOURSELF
arguments: chunk, the string to convert, allowed_special="all"
```

* Converts the text `chunk` into a list of token IDs.
* `allowed_special="all"` allows all special tokens, such as `<|endoftext|>`, to be treated as tokens.
* The output is assigned to `tokens`, a list of integers, and the model input is ready.


In [ ]:
# Tokenize
tokens = # TODO: apply instance.method(string, allowed_special="all")
print("tokens:", tokens)

```python
function: np.array
arguments: tokens, a list, dtype=np.int32, the data type
```

* `np.array(tokens, dtype=np.int32)` converts the list `tokens` into a NumPy array.
* `dtype=np.int32` explicitly makes each element in the array a 32-bit integer.
* The converted array `arr` becomes easy to use for numerical processing and machine learning model input.



In [ ]:
# Convert to a NumPy array
arr = # TODO: apply function(list, dtype=specified type)
print("numpy array\n", arr)

2. Save the token sequence in `int32` format as a binary file, `.bin`


In [ ]:
sample_bin_path = "sample.bin"

```python
function: open
arguments: file=THINK_BY_YOURSELF, mode="wb", binary write mode
```

* `with open(file=..., mode="wb") as f:` opens the file corresponding to `file=...` in **binary write mode** (`"wb"`).
* When opened with `"wb"`, if the file already exists, its contents are deleted and writing starts from an empty file.
* With a `with` statement, the file is closed automatically, so `f.close()` is not needed.


In [ ]:
# Empty the output file first, deleting existing data
# TODO: with open(file=...., mode=...) as f:
    pass

```python
function: open  
arguments: file=THINK_BY_YOURSELF, mode="ab", binary append mode
```

* `open(file=..., mode="ab")` opens a binary file in **append binary mode**.
* In `"ab"` mode, data can be added to the end of an existing file while keeping its current contents.
* The returned binary file object is stored in `sample_bin_file` and used for later writing.


In [ ]:
sample_bin_file = # TODO: function(arguments)

```python
instance: arr, a NumPy array  
method: tofile  
arguments: binary file object, THINK_BY_YOURSELF
```

* Writes the NumPy array `arr` to a file in binary format.
* As the argument, specify the destination file object, `sample_bin_file`.
* Be careful: what gets written is the raw array data, not text.



In [ ]:
# Write the NumPy array to the binary file
# TODO: instance.method(arguments)

Actually, the `tofile` above does not necessarily write the data to the file on disk immediately.

When Python writes data to a file, it does not always write to disk right away.

First, it writes data into a “buffer,” a temporary area in memory,
and once enough data has accumulated, it writes that data to disk in a batch.

However, when processing many chunks, old chunk data in the buffer may be overwritten by new chunk data.

So we make sure each chunk is reflected on disk.

A buffer is “temporary storage for data that has not yet been written to disk,”
and `flush` is the method that forces the data stored in that buffer to be written to disk.


```python
instance: binary file object, THINK_BY_YOURSELF
method: flush  
arguments: none
```

* `flush()` immediately reflects the buffered write contents on disk.
* Normally, data is written automatically when `close()` is called, but when writing a large amount of data in chunks, you can use `flush()` explicitly to save intermediate progress.



In [ ]:
# TODO: instance.method(arguments)

Please close `sample_bin_file`.


In [ ]:
# TODO: THINK_BY_YOURSELF

```python
function: np.fromfile  
arguments: file=THINK_BY_YOURSELF, dtype=THINK_BY_YOURSELF
```

* `np.fromfile` reads data from a binary file and restores it as a NumPy array with the specified data type.
* By using the same `dtype` as when writing, you can read the data back correctly.
* The loaded array is stored in `arr` and used for processing and checking.


In [ ]:
# Read back from the binary file
arr = # TODO: function(arguments)
print("numpy array\n", arr)

So far, we have looked at processing one chunk of 100 characters.

From here, we will encode `fineweb.txt`, which contains hundreds of millions of characters, in chunks of 10 million characters.


🔘 **Options**: There may be extra choices. You may use the same choice more than once.

`tiktoken`　`openai`　`get_encoding`　`"gpt2"`　`encoder`　`decoder`　`encode`　`decode`,　`output_bin_path`　`input_text_path`　`"w"`　`"r"`　`"ab"`　`"wb"`　`"utf-8"`　`chunk_size`　`chunk`　`np.array`　`tokens`　`np.int32`　`np.int8`　`tofile`　`arr`　`flush()`　`finish()`　`total_tokens`


In [ ]:
input_text_path = "/content/fineweb.txt"
output_bin_path = "/content/fineweb.bin"
chunk_size = 10_000_000
encoder = ________.___________(____) # TODO: FILL
# Empty the output binary file first, deleting existing data
with open(file=______________, mode=__) as f: # TODO: FILL
    pass
# Variable that counts the total number of tokens
total_tokens = 0
# Open the text file, tokenize it, and save it as binary
with open(file=___________, mode=__, encoding=_____) as text_file:  # TODO: FILL
    with open(file=____________, mode=___) as bin_file: # TODO: FILL
        while True:
            # Read text in fixed-size chunks
            chunk = text_file.read(________) # TODO: FILL

            # Exit the loop when reading is finished
            if not chunk:
                break

            # Convert text to tokens
            tokens = ________._______(_____, allowed_special="all") # TODO: FILL

            # Convert to a NumPy array, using int32
            arr = ________(_____, dtype=______) # TODO: FILL

            # Write to the binary file
            ___._____(bin_file) # TODO: FILL

            # Make sure the write is reflected in the file
            bin_file.______ # TODO: FILL

            # Update the number of processed tokens
            ___________ += len(arr) # TODO: FILL

            # Show progress
            print(f"✅ {total_tokens:,} tokens processed...")

# Show the final result
print(f"\n🎉 Done! Saved {total_tokens:,} tokens in total.")

<details>
<summary>Click to show/hide the answer</summary>

```python
input_text_path = "/content/fineweb.txt"
output_bin_path = "/content/fineweb.bin"
chunk_size = 10_000_000
encoder = tiktoken.get_encoding("gpt2")

# Variable that counts the total number of tokens
total_tokens = 0

# Open the text file, tokenize it, and save it as binary
with open(file=input_text_path, "r", encoding="utf-8") as f:
    with open(file=output_bin_path, "ab") as out:
        while True:
            # Read text in fixed-size chunks
            chunk = f.read(chunk_size)
            
            # Exit the loop when reading is finished
            if not chunk:
                break

            # Convert text to tokens
            tokens = encoder.encode(chunk, allowed_special="all")
            
            # Convert to a NumPy array, using int32
            arr = np.array(tokens, dtype=np.int32)
            
            # Write to the binary file
            arr.tofile(out)
            
            # Make sure the write is reflected in the file
            out.flush()
            os.fsync(out.fileno())

            # Update the number of processed tokens
            total_tokens += len(arr)
            
            # Show progress
            print(f"✅ {total_tokens:,} tokens processed...")

# Show the final result
print(f"
🎉 Done! Saved {total_tokens:,} tokens in total.")
```


3. Convert `.bin` to NumPy format, `.npy`  

Index access is possible even with a binary file, but NumPy files are more common, so we convert it to a NumPy file.

Advantage: NumPy files store type information such as `int32` inside the file, so they are easier for other people to understand when they receive them.

A binary file does not contain information saying “this is `int32`,” so you need to write that in a README, the filename, or somewhere similar.


```python
function: np.fromfile
arguments: file=path to the binary file, dtype=data type, here np.int32
```

* `np.fromfile()` is a function that creates a NumPy array directly from a binary file.
* `dtype=np.int32`: reads the binary data as `int32`, a 4-byte integer.
* It is used for quickly loading raw numeric data, such as token IDs.


In [ ]:
# 1️⃣ Load raw binary as int32.
# This is numeric data, not text data, so the load on system RAM is about 3GB.
data = # TODO: apply np.fromfile(file=file path, dtype=data type)
print("Number of loaded tokens:", len(data))

```python
function: np.save
arguments: file=save destination path, arr=NumPy array to save
```

* `np.save()` is a function that saves a NumPy array in `.npy` format.
* `.npy` is NumPy's official format for efficiently storing and loading arrays.
* `file=npy_path`: the save destination file path.


In [ ]:
# 2️⃣ Save in the correct .npy format
npy_path = "/content/fineweb.npy"
# TODO: function(arguments)

4. Load with memory mapping using `np.load(..., mmap_mode="r")`


```python
function: np.load
arguments: file=path to the .npy file to load, mmap_mode="r"
```

* `np.load()` is a function that loads a NumPy array saved in `.npy` format.
* Setting `mmap_mode="r"` enables **memory mapping**.

  * This lets us read only the needed parts without loading the entire file at once, saving RAM.
* `tokens = np.load(...)` efficiently reads the token sequence, or array, from the `.npy` file.


In [ ]:
tokens = # TODO: apply np.load(file=file path, mmap_mode="r")
sample_tokens = tokens[100:200]
print(sample_tokens)

It is a NumPy array right now, but when we feed it into the model, it needs to be in `torch.tensor` format.


```python
function: torch.from_numpy
arguments: NumPy array (sample_tokens)
```

* `torch.from_numpy(sample_tokens)` converts a NumPy array into a **PyTorch tensor**.
* The output is a `torch.Tensor` with the same shape and type as the NumPy array.



In [ ]:
torch_sample_tokens = # TODO: torch.from_numpy(NumPy array)
print(torch_sample_tokens)

It is now in `torch.tensor` format. You can see that the dtype is `torch.int32`.

You may see a “not writable” warning. Since it is opened in read mode, not being writable is normal and not a problem.


PyTorch's `nn.Embedding` accepts only `torch.int64`.

So we convert from `np.int32` to `np.int64`, and then apply `torch.from_numpy`.

This converts it into `torch.int64`.


```python
original varaible: sample_tokens (np.int32)
method: astype
arguments: data type (np.int64)

new varaible = original varaible.method(arguments)
```

* `sample_tokens.astype(np.int64)` converts the data type of the NumPy array `sample_tokens` from `int32` to `int64`, a 64-bit integer.
* The original array is not changed. A new `int64` array is created.
* In PyTorch, it is common to explicitly convert a NumPy array to an integer type before converting it into a tensor with `torch.from_numpy()`.


In [ ]:
sample_tokens_int64 = # TODO: original varaible.method(arguments)
torch_sample_tokens = torch.from_numpy(sample_tokens_int64)
print(torch_sample_tokens)

**Section 1: Converting to an npy File**<label><input type="checkbox"> Mark as Done</label>


### **Section 2: Improving the Dataloader Class**


🔘 **Options**: There may be extra choices. You may use the same choice more than once.

`np`　`torch`　`load`　`open`　`"r"`　`npy_path`　`0`　`split_index`　`np.random`　`randint`　`np.int64`　`np.int32`　`np.float32`



In [ ]:
import torch
########## NEW ##########
import numpy as __ # TODO: FILL
########## NEW ##########

class DataLoader:
    def __init__(self, npy_path, config):
        # Open the large tokenized file (.npy) with memory mapping,
        # and randomly read only the needed parts.
        self.config = config  # Model settings, such as batch_size and seq_len

        self.encoder = tiktoken.get_encoding("gpt2")
        self.vocab_size = self.encoder.n_vocab

        ########## NEW ##########
        # Open the huge tokenized .npy file with memory mapping
        self.data = ____.____(______, mmap_mode="r")  # TODO: FILL
        ########## NEW ##########

        # Decide the data ranges for training and validation. The actual data is shared
        self.train_data, self.val_data = self.split_data()

    def split_data(self):
        # Split the data 90%:10% and store it as index ranges
        split_index = int(0.9 * len(self.data))
        ########## NEW ##########
        return (__, _________), (_________, len(self.data))  # TODO: FILL
        ########## NEW ##########

    def get_batch(self, split):

        # Randomly create a batch from the specified split, 'train' or 'val'.
        # From a huge file that does not fit in memory, read only the needed slices.

        ########## NEW ##########
        # Get the data range for the chosen split
        range_start, range_end = (
            self.train_data if split == 'train' else self.val_data
        )

        # Choose random start positions
        start_indices = _________._______(
            range_start,
            range_end - self.config.input_sequence_length - 1,
            size=self.config.batch_size
        )

        # Extract continuous token sequences from each start position and batch them
        input_sequences = torch.stack([
            torch.from_numpy(
                self.data[start:start + self.config.input_sequence_length].astype(________)
            )
            for start in start_indices
        ])

        # Use the token one step ahead as the target
        target_sequences = torch.stack([
            torch.from_numpy(
                self.data[start + 1:start + self.config.input_sequence_length + 1].astype(________)
            )
            for start in start_indices
        ])
        ########## NEW ##########

        # Transfer to a device such as GPU
        return (
            input_sequences.to(self.config.device_type),
            target_sequences.to(self.config.device_type)
        )

<details>
<summary>Click to show/hide the answer</summary>

```python
import torch
########## NEW ##########
import numpy as np
########## NEW ##########

class DataLoader:
    def __init__(self, npy_path, config):
        # Open the large tokenized file (.npy) with memory mapping,
        # and randomly read only the needed parts.
        self.config = config  # Model settings, such as batch_size and seq_len

        self.encoder = tiktoken.get_encoding("gpt2")
        self.vocab_size = self.encoder.n_vocab

        ########## NEW ##########
        # Open the huge tokenized .npy file with memory mapping
        self.data = np.load(npy_path, mmap_mode="r")
        ########## NEW ##########

        # Decide the data ranges for training and validation. The actual data is shared.
        self.train_data, self.val_data = self.split_data()

    def split_data(self):
        # Split the data 90%:10% and store it as index ranges
        split_index = int(0.9 * len(self.data))
        ########## NEW ##########
        return (0, split_index), (split_index, len(self.data))
        ########## NEW ##########

    def get_batch(self, split):
        
        # Randomly create a batch from the specified split, 'train' or 'val'.
        # From a huge file that does not fit in memory, read only the needed slices.

        ########## NEW ##########
        # Get the data range for the chosen split
        range_start, range_end = (
            self.train_data if split == 'train' else self.val_data
        )

        # Choose random start positions
        start_indices = np.random.randint(
            range_start,
            range_end - self.config.input_sequence_length - 1,
            size=self.config.batch_size
        )

        # Extract continuous token sequences from each start position and batch them
        # PyTorch's `nn.Embedding` accepts only torch.int64.
        # If we apply torch.from_numpy after converting to np.int64, it becomes torch.int64.
        input_sequences = torch.stack([
            torch.from_numpy(
                self.data[start:start + self.config.input_sequence_length].astype(np.int64)
            )
            for start in start_indices
        ])

        # Use the token one step ahead as the target
        # PyTorch's `nn.Embedding` accepts only torch.int64, so we need to convert to np.int64.
        target_sequences = torch.stack([
            torch.from_numpy(
                self.data[start + 1:start + self.config.input_sequence_length + 1].astype(np.int64)
            )
            for start in start_indices
        ])
        ########## NEW ##########

        # Transfer to a device such as GPU
        return (
            input_sequences.to(self.config.device_type),
            target_sequences.to(self.config.device_type)
        )
```


**Section 2: Improving the Dataloader Class**<label><input type="checkbox"> Mark as Done</label>


Everything else is the same as the previous Chapter.


In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        # Define an embedding table with vocabulary size x embedding dimension
        self.token_embedding_table = nn.Embedding(vocab_size, embedding_dim)

    def embed(self, input_indices):
        # Get the embedding vectors corresponding to the input indices
        return self.token_embedding_table.forward(input_indices)

class PositionEmbedding(nn.Module):
    def __init__(self, input_sequence_length = 8, embedding_dim = 8):
        super().__init__()
        # Position embedding layer
        self.position_embedding_layer = nn.Embedding(input_sequence_length, embedding_dim)

    def forward(self, input_indices):
        # Shape of input tensor input_indices: [batch size, sequence length].
        sequence_length = input_indices.shape[1]

        # Create position indices according to the sequence length, such as [0, 1, 2, ..., sequence_length-1]
        position_indices = torch.arange(sequence_length, device=input_indices.device)

        # Get the embedding vectors for the position indices
        position_embeddings = self.position_embedding_layer.forward(position_indices)

        return position_embeddings

class EmbeddingModule(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        # Embedding layer for each token
        self.token_embedding_layer = TokenEmbedding(vocab_size = vocab_size, embedding_dim = config.embedding_dim)  # Token embedding layer
        self.position_embedding_layer = PositionEmbedding(input_sequence_length = config.input_sequence_length, embedding_dim = config.embedding_dim)  # Embed position information

    def forward(self, input_indices):
        # Get token embeddings
        token_embeddings = self.token_embedding_layer.embed(input_indices)

        # Get position embeddings
        position_embeddings = self.position_embedding_layer.forward(input_indices)

        # Add token embeddings and position embeddings
        embeddings = position_embeddings + token_embeddings
        return embeddings


class AttentionHead(nn.Module):
    def __init__(self, head_size, config):
        super().__init__()
        self.key_fc= nn.Linear(config.embedding_dim, head_size, bias=False)
        self.query_fc = nn.Linear(config.embedding_dim, head_size, bias=False)
        self.value_fc = nn.Linear(config.embedding_dim, head_size, bias=False)

        # Dropout
        self.dropout = nn.Dropout(config.dropout_rate)
        self.head_size = head_size

    def forward(self, input_tensor):
        B, T, C = input_tensor.shape  # batch, token length, embedding channels

        Key = self.key_fc.forward(input_tensor)     # (B, T, head_size)
        Query = self.query_fc.forward(input_tensor)   # (B, T, head_size)
        Value = self.value_fc.forward(input_tensor)   # (B, T, head_size)

        # Computing attention scores (QK^T) / sqrt(embedding_dim)
        attention_weights_before_mask = Query @ Key.transpose(-2, -1) * self.head_size**(-0.5)

        # Mask applied
        mask = torch.triu(torch.ones(T, T), diagonal=1).to(input_tensor.device)
        masked_attention_weights = attention_weights_before_mask.masked_fill(mask == 1, float('-inf'))

        # Softmax → Dropout → weighted sum
        attention_weights = F.softmax(masked_attention_weights, dim=-1)
        attention_weights = self.dropout(attention_weights)

        out = attention_weights @ Value  # (B, T, head_size)
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.num_attention_heads = config.num_attention_heads
        self.embedding_dim = config.embedding_dim
        self.head_size = int(self.embedding_dim / self.num_attention_heads)

        # Manage multiple heads with ModuleList
        self.attention_heads = nn.ModuleList([
            AttentionHead(self.head_size, config)
            for _ in range(self.num_attention_heads)
        ])

        # Linear layer that mixes the outputs of each head
        self.output_projection = nn.Linear(self.embedding_dim, self.embedding_dim)

        # Output Dropout
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(self, input_tensor):
        # Get the output of each head
        # (B, T, head_size) list
        head_outputs_list = [head.forward(input_tensor) for head in self.attention_heads]

        # Concatenate all head outputs → (B, T, embedding_dim)
        concatenated = torch.cat(head_outputs_list, dim=-1)

        # Mix outputs with a linear transformation
        projected = self.output_projection.forward(concatenated)

        # Apply Dropout to the final output
        output = self.dropout.forward(projected)

        return output

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.embedding_dim, config.hidden_dim),
            nn.ReLU(),
            nn.Linear(config.hidden_dim, config.embedding_dim),
            nn.Dropout(config.dropout_rate),
        )

    def forward(self, input_tensor):
        return self.net(input_tensor)

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

        # Each LayerNorm has its own beta and gamma.
        self.layer_norm1 = nn.LayerNorm(config.embedding_dim)
        self.layer_norm2 = nn.LayerNorm(config.embedding_dim)

        self.multihead_attention = MultiHeadAttention(config=config)
        self.feed_forward = FeedForward(config=config)

    def forward(self, input_tensor):
        # The forward method is omitted.
        normed_input = self.layer_norm1(input_tensor) # Apply layer norm to the input
        attention_output = self.multihead_attention(normed_input) # Apply multi-head attention
        residual_attention = attention_output + input_tensor # Add "before! layernorm1"
        normed_attention = self.layer_norm2(residual_attention) # Apply LayerNorm again to the residual output
        feedforward_output = self.feed_forward(normed_attention) # Apply the feed-forward network
        final_output = feedforward_output + residual_attention # Add "before" layernorm2!

        return final_output

class VocabularyLogits(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        # Layer normalization
        self.output_norm = nn.LayerNorm(config.embedding_dim)
        # Vocabulary-size projection
        self.vocab_projection = nn.Linear(config.embedding_dim, vocab_size)

    def forward(self, transformer_block_output):
        # Apply layer normalization to the Transformer block output.
        normalized_output = self.output_norm.forward(transformer_block_output)  # (B, T, C)

        # Convert scores to the vocabulary dimension with a linear layer.
        vocab_logits = self.vocab_projection.forward(normalized_output)  # (B, T, V)

        return vocab_logits

class nanoGPT(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.config = config  # Keep this because it is also used during generation.
        self.embedding = EmbeddingModule(vocab_size, config=config)
        self.blocks = nn.Sequential(*[TransformerBlock(config=config) for _ in range(config.layer_count)])
        self.vocab_projection = VocabularyLogits(vocab_size=vocab_size, config=config)
        self.criterion = nn.CrossEntropyLoss()

    # Generate text
    def generate(self, input_indices, max_new_tokens):
        # Generate only the specified number of tokens, max_new_tokens
        for _ in range(max_new_tokens):
            input_conditioned = input_indices[:, -self.config.input_sequence_length:] # Crop the input

            # The forward pass returns `(likelihood, loss)`; keep only `likelihood` as `logits`.
            logits, _ = self.forward(input_conditioned, target_indices=None)
            last_logits = logits[:, -1, :] # Extract the logits for the last token
            probs = F.softmax(last_logits, dim=-1) # Convert likelihoods to probabilities with Softmax

            # Sample the next token
            next_token = torch.multinomial(probs, num_samples=1)

            # Combine the new token and update input_indices.
            input_indices = torch.cat((input_indices, next_token), dim=1)

        # Return the final `input_indices`. Its length is the original `input_indices` plus `max_new_tokens`
        return input_indices

    # Compute likelihood and loss
    def forward(self, input_indices, target_indices):
        embeddings = self.embedding(input_indices)
        blocks_output = self.blocks(embeddings)
        logits = self.vocab_projection(blocks_output)

        # During inference, there is no target, so loss is None
        # Only probabilities, logits, are returned.
        if target_indices is None:
            return logits, None

        batch_size, token_len, vocab_size = logits.shape
        logits = logits.view(batch_size * token_len, vocab_size)
        targets = target_indices.view(batch_size * token_len)
        loss = self.criterion(logits, targets)

        return logits, loss

Define the `Trainer` class. This is exactly the same as Chapter 16.


In [ ]:
import time

class Trainer:
    def __init__(self, model, optimizer, data_loader, config):
        self.model = model
        self.optimizer = optimizer
        self.data_loader = data_loader
        self.config = config

        self.steps = []
        self.train_losses = []
        self.val_losses = []
        self.total_seen_tokens_list = []
        self.total_train_time_list = []

    def train_step(self):
        # Get a training batch.
        input_batch, target_batch = self.data_loader.get_batch('train')
        self.optimizer.zero_grad()

        # Model forward pass and loss calculation
        logits, loss = self.model(input_batch, target_batch)
        loss.backward()  # Backpropagation
        self.optimizer.step()  # Parameter update

        return loss.item() # Return the loss value

    def evaluate(self):
        self.model.eval()  # Switch to evaluation mode
        losses = {"train": [], "val": []} # Compute loss for both training and validation data
        with torch.no_grad():
            for split in ['train', 'val']:
                for _ in range(self.config.evaluation_loops):
                    input_batch, target_batch = self.data_loader.get_batch(split)
                    _, loss = self.model(input_batch, target_batch)
                    losses[split].append(loss.item())
        self.model.train()  # Return to training mode

        # Compute and return the average loss for each dataset, train and val
        return {split: sum(values) / len(values) for split, values in losses.items()}

    def train(self):
        # Run train_step for the number of times specified by config plus 1.
        for step in range(self.config.total_training_steps+1):
            # Evaluate every 100 steps.
            if step % self.config.evaluation_frequency == 0:
                if step == 0:
                  tokens_per_second = None
                  total_train_time = 0
                else:
                  current_eval_start_time = time.time()
                  evaluation_interval = current_eval_start_time - last_eval_end_time
                  total_train_time += evaluation_interval
                  tokens_per_evaluation_interval = self.config.batch_size * self.config.input_sequence_length * self.config.evaluation_frequency
                  tokens_per_second = tokens_per_evaluation_interval / evaluation_interval

                eval_loss = self.evaluate()
                total_seen_tokens = self.config.batch_size * self.config.input_sequence_length * step

                print(
                    f"step {step:05d} | "
                    f"train loss {eval_loss['train']:.4f} | "
                    f"val loss {eval_loss['val']:.4f} | "
                    f"tok/s {int(tokens_per_second) if tokens_per_second is not None else 'None'} | "
                    f"tokens {total_seen_tokens:,} | "
                    f"time {total_train_time:.2f}s"
                )

                self.steps.append(step)
                self.train_losses.append(eval_loss['train'])
                self.val_losses.append(eval_loss['val'])
                self.total_seen_tokens_list.append(total_seen_tokens)
                self.total_train_time_list.append(total_train_time)

                # Record the time this evaluation ended. The time difference to the next evaluation start becomes `evaluation_interval`.
                last_eval_end_time = time.time()

            # One training step, the main process run each time
            train_loss = self.train_step()

In [ ]:
# Configuration class that stores model settings
class ModelConfig:
    batch_size = 16
    input_sequence_length = 512  # Use longer sequences at once to reduce the relative time spent on data loading.
    ########## NEW ##########
    total_training_steps = 30_000  # Takes about 6 hours
    ########## NEW ##########
    device_type = 'cuda'  # Fix the device to GPU
    evaluation_frequency = 100  # Frequency of model performance evaluation
    learning_rate = 0.001  # Learning rate
    evaluation_loops = 10  # Number of loops during evaluation
    embedding_dim = 512  # Embedding layer size, the number of feature vector dimensions
    hidden_dim = 2048
    num_attention_heads = 8  # Number of attention heads
    layer_count = 4  # Number of model layers
    dropout_rate = 0.1  # Dropout probability
    random_seed_value = 1337  # Random seed for reproducibility

In [ ]:
# Load settings and set the seed
config = ModelConfig()
torch.manual_seed(config.random_seed_value)  # Set the random seed for reproducibility

Please create a `Dataloader` instance.



In [ ]:
########## NEW ##########
data_loader = # TODO: class(instance)
########## NEW ##########

In [ ]:
# Initialize the model and optimizer
model = nanoGPT(vocab_size = data_loader.vocab_size, config = config).to(config.device_type)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

In [ ]:
# Display the number of model parameters
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

If you log in to Google Drive beforehand,

you can run everything without stopping until the cell that saves the logs.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

If Google Colab has no user interaction for more than 2 hours, it may treat the session as idle and disconnect it.

So we will print output in the background.

A background thread keeps running `print` every 10 minutes, so Colab recognizes that “a thread is still processing = the user is working,” which helps prevent session disconnection.

Note: This method cannot increase your GPU credits.

It only helps prevent the session from stopping when you are not interacting with it.


```python
instance: thread = threading.Thread(arguments 1, arguments 2)
arguments 1: target = the function to run in the thread
arguments 2: daemon = whether to run it in the background as a daemon thread
```

* `threading.Thread` is a class for creating a new thread.
* With `target=keep_alive`, the `keep_alive` function runs inside the thread.
* Setting `daemon=True` makes it a background thread that automatically ends when the program's main thread ends.



In [ ]:
import time
import threading

def keep_alive():
    while True:
        print("Keeping session alive...")
        time.sleep(600)  # Print every 10 minutes

# Create a background thread instance to run the keep_alive function
thread = # TDOO: class(arguments)

```python
instance: thread
method: start
arguments: none
```

* Calling `thread.start()` starts the thread, and the specified `target` function, in this case `keep_alive`, runs asynchronously.
* By using `.start()`, you can run processing in parallel with the main thread.


In [ ]:
# Start the background thread
# TODO: instance.method(arguments)
print("Keeping the session alive in the background...")

### 🚀 Training Start!

> ⚠️ (2026/1/23) We are trying to prevent session disconnection in the background, but it seems the session still disconnects after about 2 hours.<br>
> We are currently looking for a workaround. Until we find one, please either pay for Colab Pro or lower total steps to 8,000.<br>
> Add a new code cell, run `config.total_training_steps = 8000`, and then start training.<br>

This time, we use a **T4 GPU with 15 GB RAM**, with
`batch size = 16` and `total steps = 30,000`.

Training takes about 6 hours.

Run **all** cells after this point, then leave it alone for 6 hours.

The cells that save logs and the model will wait to run,

and they will execute after the training cell finishes.





In [ ]:
print("===Training started successfully===")

# Train the model
trainer = Trainer(model, optimizer, data_loader, config)
trainer.train()

Let's plot it with `matplotlib`, using `Step` on the horizontal axis and `Loss` on the vertical axis.


In [ ]:
# Draw the graph.
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.plot(trainer.steps, trainer.train_losses, label='Train Loss')
plt.plot(trainer.steps, trainer.val_losses, label='Validation Loss')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training and Validation Loss over Steps')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Try inference too
# Switch to evaluation mode. Disable Dropout.
model.eval()
print("Model set to eval mode")

In [ ]:
prompt = "New York is"
print(f"\nInput prompt: {prompt}")

encoder = tiktoken.get_encoding("gpt2")

# Tokenize and convert to a tensor
encoded = encoder.encode(prompt, allowed_special="all") # Encode text into IDs
print("encoded", encoded)
encoded_tensor = torch.tensor(encoded, dtype=torch.long) # Convert the ID list into tensor format
print_formatted_tensor(encoded_tensor)
encoded_tensor = encoded_tensor.unsqueeze(0)  # Add batch dimension
print_formatted_tensor(encoded_tensor)
encoded_tensor = encoded_tensor.to(config.device_type) # Transfer encoded_tensor to cuda (GPU)
print_formatted_tensor(encoded_tensor)

In [ ]:
# Text generation
generated_text = model.generate(encoded_tensor, max_new_tokens=512)

#### **The Most Important Checkpoint After Training**

Let's check the output. The loss should be around 4.0 to 4.1.

You should see that the output is relatively text-like.

The context still jumps around and feels unnatural, but it has some theme consistency.

Before changing the tokenizer, the loss was 1.0 to 1.1, but the actual quality of the generated text feels better now.

The loss is larger simply because the classification problem became harder. When we tokenized one character at a time, the model basically only had to predict the next token from the 26 letters of the alphabet. Now it predicts from about 50,000 types of subwords. Of course, many of those 50,000 types are rarely used, but the classification difficulty naturally increases. As a result, the loss inevitably becomes larger.

---

By adding compute resources, the model's performance has steadily improved.

However, if you look at the graph, it is already getting close to a `loss plateau`, so further improvement is limited.

We could make the model even larger to lower the `loss plateau`, but that would require more compute resources, so it is a tricky trade-off.

This trade-off leads into the famous scaling laws.


In [ ]:
decoded_text = encoder.decode(generated_text[0].tolist())
print(decoded_text)

We are using plenty of T4 GPU time again, so let's keep the training logs properly.

Google Drive is recommended.


In [ ]:
# Collect logs from the trained trainer
results = {
    "step": trainer.steps,
    "train_loss": trainer.train_losses,
    "val_loss": trainer.val_losses,
    "total_seen_tokens": trainer.total_seen_tokens_list,
    "total_train_time": trainer.total_train_time_list,
}

print(results)

In [ ]:
import pandas as pd
# Convert to a pandas DataFrame
df = pd.DataFrame(results)

In [ ]:
df

In [ ]:
# Create the destination folder
import os
dir_path = "/content/drive/MyDrive/nanoGPT_logs/Chapter19"
os.makedirs(dir_path, exist_ok=True)

In [ ]:
# Specify the destination path for saving as a CSV file.
save_path = "/content/drive/MyDrive/nanoGPT_logs/Chapter19/training_logs.csv"

In [ ]:
# Save as CSV
df.to_csv(save_path, index=False)
print(f"✅ CSV saved to: {save_path}")

In [ ]:
# Convert class attributes to a dictionary
config_class_dict = vars(config.__class__)
print(config_class_dict)

In [ ]:
# Get the (key, value) pairs of the dictionary
config_dict_items = config_class_dict.items()
print(config_dict_items)

In [ ]:
config_dict = {
    key: value
    for key, value in config_dict_items
    if not key.startswith("__")
}

print(config_dict)

In [ ]:
# Create the destination file path.
# dir_path = "/content/drive/MyDrive/nanoGPT_logs/Chapter17"
config_path = os.path.join(dir_path, "model_config.json")
print(config_path)

In [ ]:
import json
# Open the file in write mode (w) so it can be handled with variable `f`
with open(config_path, "w") as f:
    json.dump(config_dict, f)

print(f"✅ Config saved to: {config_path}")

In [ ]:
model_path = os.path.join(dir_path, "model.pt")
print(model_path)

In [ ]:
torch.save(model.state_dict(), model_path)
print(f"✅ Model saved to: {model_path}")

**Section 3: Long Training** <label><input type="checkbox"> Mark as Done</label>


**⚠️ Disconnect the runtime from the 🔽 in the top right to stop using credits.** <label><input type="checkbox">Disconnected</label>


**Chapter 19: Long Train** <label><input type="checkbox"> Mark as Done</label>